<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_4_transfer_learning.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_4_transfer_learning.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F03_vision%2Fnotebooks%2Fsesion_03_4_transfer_learning.ipynb)

# SI7011 · Sesión 03 · 4. Transfer learning
**Juan David Martínez Vargas · Universidad EAFIT**

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | pocas imágenes por clase: 5, 20 y todas | el problema |
| 2 | *linear probe*: backbone congelado, solo la cabeza aprende | accuracy con 5 imágenes por clase |
| 3 | ResNet-18 desde cero | lo que cuesta no tener preentrenamiento |
| 4 | *fine-tuning* con $\eta$ distinto para backbone y cabeza | cuándo vale la pena descongelar |
| 5 | preprocesamiento equivocado | un error silencioso |
| 6 | las tres estrategias frente a la cantidad de datos | la figura de la diapositiva |
| 7 | qué aprendió el backbone | kernels y parches que más activan cada canal |

**Datos:** Oxford-IIIT Pets, 37 razas de perros y gatos, unas 100 imágenes por raza. **Usa GPU** (Colab o Kaggle): unos 15 minutos. En CPU la sección 3 y la 4 tardan mucho más.

In [ ]:
import copy, json, math, time
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
int_epochs = lambda: plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
print('PyTorch', torch.__version__, '· device', device)

## Datos: Oxford-IIIT Pets
Las imágenes tienen tamaños distintos. Las llevamos una sola vez a **160×160** (redimensionar el lado corto a 176 y recortar el centro) y las guardamos como enteros de 8 bits: unas 7 300 imágenes ocupan unos 560 MB. La ResNet-18 se entrenó en ImageNet con 224×224, pero gracias al *global average pooling* acepta 160×160 y entrena casi dos veces más rápido.

La partición oficial `trainval` se divide en entrenamiento y validación (20 imágenes por raza), y `test` queda para el final.

In [ ]:
resize = transforms.Compose([transforms.Resize(176), transforms.CenterCrop(160), transforms.PILToTensor()])
trainval = datasets.OxfordIIITPet('data', split='trainval', target_types='category', download=True, transform=resize)
test_set = datasets.OxfordIIITPet('data', split='test', target_types='category', download=True, transform=resize)
CLASSES = trainval.classes

def to_tensors(ds):
    t0 = time.time()
    X = torch.stack([ds[i][0] for i in range(len(ds))])            # uint8, N × 3 × 160 × 160
    y = torch.tensor([ds[i][1] for i in range(len(ds))])
    print(f'{len(ds)} imágenes en {time.time() - t0:.0f} s')
    return X, y

X_tv, y_tv = to_tensors(trainval)
X_test, y_test = to_tensors(test_set)

In [ ]:
g = torch.Generator().manual_seed(0)
val_idx, train_by_class = [], {}
for c in range(len(CLASSES)):
    idx = (y_tv == c).nonzero().squeeze(1)
    idx = idx[torch.randperm(len(idx), generator=g)]
    val_idx.append(idx[:20]); train_by_class[c] = idx[20:]       # 20 por raza para validar
val_idx = torch.cat(val_idx)
X_val, y_val = X_tv[val_idx], y_tv[val_idx]
print('entrenamiento por raza:', min(len(v) for v in train_by_class.values()), '–', max(len(v) for v in train_by_class.values()),
      '· validación', len(val_idx), '· prueba', len(y_test))

fig, axes = plt.subplots(1, 8, figsize=(11, 1.8))
for ax, i in zip(axes, val_idx[::93][:8]):
    ax.imshow(X_tv[i].permute(1, 2, 0)); ax.set_title(CLASSES[y_tv[i]], fontsize=7); ax.axis('off')
plt.show()

## 1. Pocas imágenes por clase
`subset(k)` toma las primeras $k$ imágenes de entrenamiento de cada raza. Compararemos $k = 5$, $k = 20$ y todas (unas 80 por raza).

In [ ]:
def subset(k=None):
    idx = torch.cat([v[:k] if k else v for v in train_by_class.values()])
    return X_tv[idx], y_tv[idx]

KS = {'5': 5, '20': 20, 'todas': None}
{name: len(subset(k)[1]) for name, k in KS.items()}

## Herramientas: normalizar, aumentar, entrenar
La red preentrenada espera imágenes normalizadas con la media y la desviación estándar **de ImageNet**, no las de nuestros datos. Las tomamos de `weights.transforms()`, que es la receta oficial con la que se entrenó (diapositiva *Match the pretraining preprocessing*).

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
preset = weights.transforms()
print(preset)
IMNET_MEAN = torch.tensor(preset.mean, device=device).view(1, 3, 1, 1)
IMNET_STD = torch.tensor(preset.std, device=device).view(1, 3, 1, 1)
prep = lambda xb: (xb.to(device).float() / 255 - IMNET_MEAN) / IMNET_STD    # uint8 → normalizado, en el device

def augment(xb, pad=8):                                    # el mismo del notebook 3, para 160×160
    flip = torch.rand(len(xb), device=xb.device) < 0.5
    xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
    padded = F.pad(xb, (pad, pad, pad, pad), mode='reflect')
    ij = torch.randint(0, 2 * pad + 1, (len(xb), 2))
    return torch.stack([padded[k, :, i:i + 160, j:j + 160] for k, (i, j) in enumerate(ij.tolist())])

In [ ]:
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, X, y, chunk=256):
    model.eval()
    logits = torch.cat([model(prep(X[i:i + chunk])).cpu() for i in range(0, len(X), chunk)])
    return loss_fn(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, X, y, opt, epochs=10, batch_size=64, sched=None, before_epoch=None):
    hist = {'loss': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        if before_epoch is not None:
            before_epoch(model)
        perm, total = torch.randperm(len(X)), 0.0
        for i in range(0, len(X), batch_size):
            b = perm[i:i + batch_size]
            loss = loss_fn(model(augment(prep(X[b]))), y[b].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            if sched is not None:
                sched.step()
            total += loss.item() * len(b)
        hist['loss'].append(total / len(X)); hist['val_acc'].append(evaluate(model, X_val, y_val)[1])
    print(f"{len(X)} imágenes · época {epochs}: loss {hist['loss'][-1]:.3f} · val_acc {hist['val_acc'][-1]:.3f} · {time.time() - t0:.0f} s")
    return hist

## 2. Linear probe
Congelamos el backbone $g$ y entrenamos solo una capa lineal sobre sus features (diapositiva *Linear probe*):
$$\hat y = \operatorname{softmax}\big(W\, g(x) + b\big)$$
Como $g$ no cambia, las features se calculan **una sola vez** y se guardan. Reemplazar `fc` por `nn.Identity()` deja la salida del *global average pooling*: 512 números por imagen.

In [ ]:
backbone = resnet18(weights=weights)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()

@torch.no_grad()
def features(X, normalize=prep, chunk=256):
    return torch.cat([backbone(normalize(X[i:i + chunk])).cpu() for i in range(0, len(X), chunk)])

t0 = time.time()
G_tv, G_val, G_test = features(X_tv), features(X_val), features(X_test)
print(f'features {tuple(G_tv.shape)} en {time.time() - t0:.0f} s')

In [ ]:
def probe(k=None, G=None, G_v=None, steps=300):
    G, G_v = (G_tv, G_val) if G is None else (G, G_v)
    idx = torch.cat([v[:k] if k else v for v in train_by_class.values()])
    head = nn.Linear(512, len(CLASSES))
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-4)
    for _ in range(steps):                                    # lote completo: son pocas filas de 512 números
        loss = loss_fn(head(G[idx]), y_tv[idx]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head(G_v).argmax(1) == y_val).float().mean().item(), head

acc_probe = {}
for name, k in KS.items():
    t0 = time.time()
    acc_probe[name], _ = probe(k)
    print(f'linear probe · k = {name:5s} · val_acc {acc_probe[name]:.3f} · {time.time() - t0:.1f} s')

Con solo **5 imágenes por raza** la linear probe ya distingue buena parte de las 37 razas. Cada probe se entrena en segundos, porque el trabajo caro, pasar las imágenes por la ResNet, se hizo una sola vez.

## 3. ResNet-18 desde cero
**Cambia los pesos:** `resnet18(weights=None)`. Es la misma arquitectura, con la inicialización aleatoria, y todo se entrena: 11 millones de parámetros.

In [ ]:
acc_scratch, h_scratch = {}, {}
for name, k in KS.items():
    torch.manual_seed(0)
    model = resnet18(weights=None, num_classes=len(CLASSES)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    h_scratch[name] = fit(model, *subset(k), opt, epochs=15)
    acc_scratch[name] = max(h_scratch[name]['val_acc'])

Con 5 o 20 imágenes por raza, la red desde cero apenas supera el azar ($1/37 \approx 0.027$) y, aun con todas, queda muy por debajo de la linear probe. La arquitectura es la misma: la diferencia está en lo que el backbone ya aprendió en ImageNet.

## 4. Fine-tuning
**Cambia qué se entrena:** partimos de los pesos de ImageNet, ponemos una cabeza nueva de 37 salidas y entrenamos **todo**, con cuidado (diapositiva *Fine-tuning*):

- dos **grupos de parámetros**: $\eta = 10^{-4}$ para el backbone y $\eta = 10^{-3}$ para la cabeza, que empieza aleatoria;
- *warmup* de una época y decaimiento coseno, el `LambdaLR` de la sesión 02;
- BatchNorm del backbone en modo `eval()`: con lotes de 64 imágenes sus estadísticas de ImageNet son más confiables que las del lote.

In [ ]:
def finetune_model():
    model = resnet18(weights=weights)
    model.fc = nn.Linear(512, len(CLASSES))                   # cabeza nueva, aleatoria
    return model.to(device)

def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

def warmup_cosine(opt, steps_per_epoch, epochs):
    total, warm = steps_per_epoch * epochs, steps_per_epoch
    f = lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

model = finetune_model()
backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
opt = torch.optim.AdamW([{'params': backbone_params, 'lr': 1e-4},
                         {'params': model.fc.parameters(), 'lr': 1e-3}], weight_decay=0.05)
[g['lr'] for g in opt.param_groups]

In [ ]:
EPOCHS_FT = 8
acc_ft, h_ft, models_ft = {}, {}, {}
for name, k in KS.items():
    torch.manual_seed(0)
    model = finetune_model()
    backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
    opt = torch.optim.AdamW([{'params': backbone_params, 'lr': 1e-4},
                             {'params': model.fc.parameters(), 'lr': 1e-3}], weight_decay=0.05)
    Xk, yk = subset(k)
    sched = warmup_cosine(opt, math.ceil(len(Xk) / 64), EPOCHS_FT)
    h_ft[name] = fit(model, Xk, yk, opt, epochs=EPOCHS_FT, sched=sched, before_epoch=freeze_bn)
    acc_ft[name], models_ft[name] = max(h_ft[name]['val_acc']), model

In [ ]:
for name in KS:
    print(f'k = {name:5s} · probe {acc_probe[name]:.3f} · fine-tuning {acc_ft[name]:.3f} · desde cero {acc_scratch[name]:.3f}')

Con muchas imágenes, descongelar el backbone suele ganarle a la probe, porque las features se ajustan a las razas. Con muy pocas la ventaja se reduce o se invierte: hay demasiados parámetros para tan pocos ejemplos. Compara las columnas fila por fila antes de seguir.

## 5. Un error silencioso: preprocesamiento equivocado
**Cambia una línea:** calculamos las features sin la normalización de ImageNet, solo dividiendo por 255. El código corre igual y no hay ningún error.

In [ ]:
G_tv_bad = features(X_tv, normalize=lambda xb: xb.to(device).float() / 255)
G_val_bad = features(X_val, normalize=lambda xb: xb.to(device).float() / 255)
acc_bad, _ = probe(None, G_tv_bad, G_val_bad)
print(f"linear probe, todas · con normalización de ImageNet {acc_probe['todas']:.3f} · sin ella {acc_bad:.3f}")

La accuracy baja (cuánto, depende del backbone), pero nada en el código lo señala. En producción este error aparece cuando el preprocesamiento del despliegue no es el mismo del entrenamiento: por eso en el integrador la normalización viaja **dentro** del artefacto.

## 6. Las tres estrategias frente a la cantidad de datos
Es la figura de la diapositiva, medida en validación. Los resultados se guardan en `resultados_transfer.json`.

In [ ]:
x = [5, 20, sum(len(v) for v in train_by_class.values()) / len(CLASSES)]
for name, acc, style in [('linear probe', acc_probe, 'o-'), ('fine-tuning', acc_ft, 's-'), ('desde cero', acc_scratch, '^-')]:
    plt.plot(x, [acc[k] for k in KS], style, label=name)
plt.axhline(1 / len(CLASSES), color='gray', ls=':', label='azar')
plt.xscale('log'); plt.xticks(x, ['5', '20', f'{x[-1]:.0f} (todas)']); plt.minorticks_off()
plt.xlabel('imágenes de entrenamiento por raza'); plt.ylabel('accuracy de validación'); plt.legend(); plt.show()
json.dump({'imagenes_por_clase': x, 'probe': acc_probe, 'fine_tuning': acc_ft, 'desde_cero': acc_scratch},
          open('resultados_transfer.json', 'w'), indent=1)

## 7. Qué aprendió el backbone
Los 64 kernels de la primera capa de la ResNet-18 de ImageNet son de 7×7 y se pueden ver como imágenes en color. Compáralos con los de 3×3 que aprendió la CNN pequeña en el notebook 2.

In [ ]:
W = backbone.conv1.weight.detach().cpu()                    # 64 × 3 × 7 × 7
W = (W - W.amin(dim=(1, 2, 3), keepdim=True)) / (W.amax(dim=(1, 2, 3), keepdim=True) - W.amin(dim=(1, 2, 3), keepdim=True))
fig, axes = plt.subplots(4, 16, figsize=(11, 3))
for ax, w in zip(axes.flat, W):
    ax.imshow(w.permute(1, 2, 0)); ax.axis('off')
plt.suptitle('kernels de la primera capa (7×7×3)', fontsize=9); plt.show()

Más adentro los canales responden a patrones más grandes. Para cada etapa buscamos, entre las imágenes de validación, los parches que más activan algunos canales. El parche es la región de la entrada alrededor de la posición de máxima activación, de un tamaño parecido al campo receptivo de esa etapa.

In [ ]:
stages = {'layer2': (backbone.layer2, 40), 'layer3': (backbone.layer3, 72), 'layer4': (backbone.layer4, 120)}
acts = {}
hooks = [m.register_forward_hook(lambda m, i, o, name=name: acts.__setitem__(name, o.detach().cpu()))
         for name, (m, _) in stages.items()]
with torch.no_grad():
    backbone(prep(X_val[:400]))
[h.remove() for h in hooks]

fig, axes = plt.subplots(3, 12, figsize=(12, 3.4))
for row, (name, (_, crop)) in enumerate(stages.items()):
    a = acts[name]                                           # N × C × h × w
    channels = a.amax(dim=(0, 2, 3)).topk(4).indices         # los 4 canales con la activación más alta
    for c_i, c in enumerate(channels):
        flat = a[:, c].flatten(1)
        top = flat.amax(1).topk(3).indices                    # las 3 imágenes que más lo activan
        for t_i, n in enumerate(top):
            pos_ = flat[n].argmax().item(); i, j = divmod(pos_, a.shape[3])
            ci, cj = int((i + 0.5) * 160 / a.shape[2]), int((j + 0.5) * 160 / a.shape[3])
            i0, j0 = max(0, min(160 - crop, ci - crop // 2)), max(0, min(160 - crop, cj - crop // 2))
            ax = axes[row, 3 * c_i + t_i]
            ax.imshow(X_val[n, :, i0:i0 + crop, j0:j0 + crop].permute(1, 2, 0)); ax.axis('off')
    axes[row, 0].set_title(name, fontsize=8, loc='left')
plt.show()

Cada grupo de tres parches corresponde a un canal. En `layer2` suelen aparecer texturas y bordes; en `layer4`, partes reconocibles como ojos, narices u orejas. Es la jerarquía de la diapositiva *A hierarchy of features*, sin haber visto ninguna etiqueta de razas.

## Evaluación en prueba
Elegimos la estrategia por validación, con todas las imágenes, y evaluamos en prueba una sola vez.

In [ ]:
if acc_ft['todas'] >= acc_probe['todas']:
    test_acc = evaluate(models_ft['todas'], X_test, y_test)[1]; chosen = 'fine-tuning'
else:
    _, head = probe(None)
    with torch.no_grad():
        test_acc = (head(G_test).argmax(1) == y_test).float().mean().item(); chosen = 'linear probe'
print(f'{chosen} · prueba: acc {test_acc:.3f}')

## Tu turno · 15 minutos
1. Haz *fine-tuning* solo de `layer4` y la cabeza: pon `requires_grad = False` en el resto. ¿Dónde queda frente a la probe y al fine-tuning completo con $k = 20$?
2. Usa el mismo $\eta = 10^{-3}$ para el backbone y la cabeza con $k = 5$. ¿Qué le pasa a la accuracy en las primeras épocas?
3. Quita `before_epoch=freeze_bn`. ¿Cambia algo con lotes de 64? ¿Y con `batch_size=8`?
4. Repite la probe con `resnet50` (`ResNet50_Weights.IMAGENET1K_V2`; las features son de 2048). ¿Vale la pena un backbone más grande con 5 imágenes por raza?

**Para pensar.** La linear probe con 5 imágenes por raza le gana a 11 millones de parámetros entrenados desde cero con todas. ¿Qué supuesto, aparte de la arquitectura, está haciendo la probe sobre nuestros datos? (pista: diapositiva *When transfer helps less*).

In [ ]:
# Experimento 1: fine-tuning solo de layer4 + cabeza

### Referencias
- Yosinski et al. (2014), *How transferable are features in deep neural networks?*, NeurIPS.
- Kornblith et al. (2019), *Do Better ImageNet Models Transfer Better?*, CVPR.
- Parkhi et al. (2012), *Cats and Dogs*, CVPR (Oxford-IIIT Pets).
- Zeiler y Fergus (2014), *Visualizing and Understanding Convolutional Networks*, ECCV (la idea de la sección 7).